In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

df = pd.read_csv("rent_acled_with_lags.csv")

df = df.dropna(subset=["середня_ціна_грн"]).copy()

if "log_rent" not in df.columns:
    df["log_rent"] = np.log(df["середня_ціна_грн"])

df["region_type"] = df["область"].astype(str) + "__" + df["тип_квартири"].astype(str)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())
print(df[["область", "month", "тип_квартири", "середня_ціна_грн", "log_rent"]].head())

Rows: 2369
Columns: ['область', 'month', 'тип_квартири', 'середня_ціна_грн', 'data_status', 'shelling_count', 'drone_count', 'battle_count', 'total_attacks', 'fatalities', 'any_attack', 'unemp_monthly', 'total_attacks_lag1', 'total_attacks_lag2', 'fatalities_lag1', 'fatalities_lag2', 'rent_lag1', 'rent_lag2', 'log_rent', 'unemp_lag1', 'region_type']
             область    month         тип_квартири  середня_ціна_грн  log_rent
0  Івано-Франківська  2022-02  1-кімнатна квартира            6620.0  8.797851
1  Івано-Франківська  2022-03  1-кімнатна квартира            6800.0  8.824678
2  Івано-Франківська  2022-04  1-кімнатна квартира            5725.0  8.652598
3  Івано-Франківська  2022-05  1-кімнатна квартира            8779.0  9.080118
4  Івано-Франківська  2022-06  1-кімнатна квартира            7831.0  8.965845


In [2]:
alpha = 0.05

def one_sided_left_test(model, var, alpha=0.05):
    coef = model.params[var]
    p_two_sided = model.pvalues[var]
    ci_low, ci_high = model.conf_int().loc[var]

    if coef < 0:
        p_one_sided = p_two_sided / 2
    else:
        p_one_sided = 1 - (p_two_sided / 2)

    print("=" * 80)
    print(f"Variable: {var}")
    print(f"Coefficient: {coef}")
    print(f"Two-sided p-value: {p_two_sided}")
    print(f"One-sided p-value for H1: beta < 0: {p_one_sided}")
    print(f"95% confidence interval: ({ci_low}, {ci_high})")

    if coef < 0 and p_one_sided < alpha:
        print("Reject H0: the results support H1.")
    else:
        print("Fail to reject H0.")
    print("=" * 80)

In [3]:
# Model 1. Baseline contemporaneous conflict model
# H0_1: total_attacks does not affect rent
# H1_1: higher total_attacks lowers rent
#
# H0_2: fatalities do not affect rent
# H1_2: higher fatalities lower rent
#
# H0_3: unemp_monthly does not affect rent
# H1_3: higher unemployment lowers rent

model_1_df = df.dropna(subset=["total_attacks", "fatalities", "unemp_monthly"]).copy()

model_1 = smf.ols(
    'log_rent ~ total_attacks + fatalities + unemp_monthly + C(область) + C(month) + C(Q("тип_квартири"))',
    data=model_1_df
).fit(cov_type="cluster", cov_kwds={"groups": model_1_df["region_type"]})

print(model_1.summary())

                            OLS Regression Results                            
Dep. Variable:               log_rent   R-squared:                       0.803
Model:                            OLS   Adj. R-squared:                  0.798
Method:                 Least Squares   F-statistic:                 2.749e+04
Date:                Wed, 15 Apr 2026   Prob (F-statistic):          1.82e-126
Time:                        00:00:38   Log-Likelihood:                 631.47
No. Observations:                2369   AIC:                            -1139.
Df Residuals:                    2307   BIC:                            -781.2
Df Model:                          61                                         
Covariance Type:              cluster                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [4]:
one_sided_left_test(model_1, "total_attacks", alpha)
one_sided_left_test(model_1, "fatalities", alpha)
one_sided_left_test(model_1, "unemp_monthly", alpha)

Variable: total_attacks
Coefficient: 4.919209933504018e-05
Two-sided p-value: 0.6573522554864999
One-sided p-value for H1: beta < 0: 0.67132387225675
95% confidence interval: (-0.00016816788692876352, 0.00026655208559884385)
Fail to reject H0.
Variable: fatalities
Coefficient: -5.0185055191460026e-05
Two-sided p-value: 0.25351609743848846
One-sided p-value for H1: beta < 0: 0.12675804871924423
95% confidence interval: (-0.00013632670535897482, 3.5956594976054785e-05)
Fail to reject H0.
Variable: unemp_monthly
Coefficient: -3.558913667292806e-07
Two-sided p-value: 0.9428358703479232
One-sided p-value for H1: beta < 0: 0.4714179351739616
95% confidence interval: (-1.008358680631765e-05, 9.371804072859089e-06)
Fail to reject H0.


In [5]:
# Model 2. Lagged conflict model
# H0_4: total_attacks_lag1 does not affect current rent
# H1_4: higher total_attacks_lag1 lowers current rent

model_2_df = df.dropna(subset=["total_attacks_lag1", "fatalities", "unemp_monthly"]).copy()

model_2 = smf.ols(
    'log_rent ~ total_attacks_lag1 + fatalities + unemp_monthly + C(область) + C(month) + C(Q("тип_квартири"))',
    data=model_2_df
).fit(cov_type="cluster", cov_kwds={"groups": model_2_df["region_type"]})

print(model_2.summary())

                            OLS Regression Results                            
Dep. Variable:               log_rent   R-squared:                       0.809
Model:                            OLS   Adj. R-squared:                  0.804
Method:                 Least Squares   F-statistic:                 1.075e+04
Date:                Wed, 15 Apr 2026   Prob (F-statistic):          3.78e-113
Time:                        00:00:38   Log-Likelihood:                 645.81
No. Observations:                2306   AIC:                            -1170.
Df Residuals:                    2245   BIC:                            -819.3
Df Model:                          60                                         
Covariance Type:              cluster                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [6]:
one_sided_left_test(model_2, "total_attacks_lag1", alpha)

Variable: total_attacks_lag1
Coefficient: 7.210014930375384e-05
Two-sided p-value: 0.5196846066785341
One-sided p-value for H1: beta < 0: 0.7401576966607329
95% confidence interval: (-0.00014738843218962546, 0.00029158873079713316)
Fail to reject H0.


In [7]:
# Model 3. Attack composition model
# H0_5: shelling_count = drone_count = battle_count
# H1_5: at least one effect differs

model_3_df = df.dropna(subset=["shelling_count", "drone_count", "battle_count", "fatalities", "unemp_monthly"]).copy()

model_3 = smf.ols(
    'log_rent ~ shelling_count + drone_count + battle_count + fatalities + unemp_monthly + '
    'C(область) + C(month) + C(Q("тип_квартири"))',
    data=model_3_df
).fit(cov_type="cluster", cov_kwds={"groups": model_3_df["region_type"]})

print(model_3.summary())

                            OLS Regression Results                            
Dep. Variable:               log_rent   R-squared:                       0.807
Model:                            OLS   Adj. R-squared:                  0.801
Method:                 Least Squares   F-statistic:                 4.253e+04
Date:                Wed, 15 Apr 2026   Prob (F-statistic):          9.14e-133
Time:                        00:00:38   Log-Likelihood:                 650.89
No. Observations:                2369   AIC:                            -1174.
Df Residuals:                    2305   BIC:                            -804.5
Df Model:                          63                                         
Covariance Type:              cluster                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [8]:
for var in ["shelling_count", "drone_count", "battle_count"]:
    print(f"{var}: coef = {model_3.params[var]}, p-value = {model_3.pvalues[var]}")

shelling_count: coef = -0.00013765241446950607, p-value = 0.14782648717463645
drone_count: coef = 0.00032069191474917345, p-value = 0.006785842974578725
battle_count: coef = -0.0007081995092683461, p-value = 8.764117529627498e-05


In [9]:
test_joint = model_3.f_test("shelling_count = drone_count, drone_count = battle_count")

print(test_joint)

pval_joint = float(test_joint.pvalue)
print("Joint test p-value:", pval_joint)

if pval_joint < alpha:
    print("Reject H0: the effects are not equal.")
else:
    print("Fail to reject H0: there is not enough evidence that the effects differ.")

<F test: F=9.887344987489044, p=0.00017825111650917824, df_denom=65, df_num=2>
Joint test p-value: 0.00017825111650917824
Reject H0: the effects are not equal.


In [10]:
# Optional Model 4. Attack occurrence dummy
# H0_6: average rent is the same in months with attacks
#       and without attacks
# H1_6: average rent is lower in months with attacks

model_4_df = df.dropna(subset=["any_attack", "fatalities", "unemp_monthly"]).copy()

model_4 = smf.ols(
    'log_rent ~ any_attack + fatalities + unemp_monthly + C(область) + C(month) + C(Q("тип_квартири"))',
    data=model_4_df
).fit(cov_type="cluster", cov_kwds={"groups": model_4_df["region_type"]})

print(model_4.summary())

                            OLS Regression Results                            
Dep. Variable:               log_rent   R-squared:                       0.803
Model:                            OLS   Adj. R-squared:                  0.798
Method:                 Least Squares   F-statistic:                 1.458e+04
Date:                Wed, 15 Apr 2026   Prob (F-statistic):          1.62e-117
Time:                        00:00:38   Log-Likelihood:                 630.85
No. Observations:                2369   AIC:                            -1138.
Df Residuals:                    2307   BIC:                            -780.0
Df Model:                          61                                         
Covariance Type:              cluster                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [11]:
one_sided_left_test(model_4, "any_attack", alpha)

Variable: any_attack
Coefficient: -0.007196209532768838
Two-sided p-value: 0.4682648079991123
One-sided p-value for H1: beta < 0: 0.23413240399955615
95% confidence interval: (-0.026642245828872084, 0.012249826763334408)
Fail to reject H0.


In [13]:
results = []

def collect_left_test_result(model, var, hypothesis_name):
    coef = model.params[var]
    p_two_sided = model.pvalues[var]

    if coef < 0:
        p_one_sided = p_two_sided / 2
    else:
        p_one_sided = 1 - (p_two_sided / 2)

    decision = "Reject H0" if (coef < 0 and p_one_sided < alpha) else "Fail to reject H0"

    results.append({
        "Hypothesis": hypothesis_name,
        "Model variable": var,
        "Coefficient": coef,
        "Two-sided p-value": p_two_sided,
        "One-sided p-value": p_one_sided,
        "Decision at 5%": decision
    })

collect_left_test_result(model_1, "total_attacks", "H1: current attacks lower rent")
collect_left_test_result(model_1, "fatalities", "H1: fatalities lower rent")
collect_left_test_result(model_1, "unemp_monthly", "H1: unemployment lowers rent")
collect_left_test_result(model_2, "total_attacks_lag1", "H1: lagged attacks lower rent")
collect_left_test_result(model_4, "any_attack", "H1: months with attacks have lower rent")

results.append({
    "Hypothesis": "H1: shelling, drone, and battle effects differ",
    "Model variable": "shelling_count = drone_count = battle_count",
    "Coefficient": np.nan,
    "Two-sided p-value": float(test_joint.pvalue),
    "One-sided p-value": np.nan,
    "Decision at 5%": "Reject H0" if float(test_joint.pvalue) < alpha else "Fail to reject H0"
})

results_df = pd.DataFrame(results)
results_df

,Hypothesis,Model variable,Coefficient,Two-sided p-value,One-sided p-value,Decision at 5%
0,H1: current attacks lower rent,total_attacks,4.919210e-05,0.657352,0.671324,Fail to reject H0
1,H1: fatalities lower rent,fatalities,-5.018506e-05,0.253516,0.126758,Fail to reject H0
2,H1: unemployment lowers rent,unemp_monthly,-3.558914e-07,0.942836,0.471418,Fail to reject H0
3,H1: lagged attacks lower rent,total_attacks_lag1,7.210015e-05,0.519685,0.740158,Fail to reject H0
4,H1: months with attacks have lower rent,any_attack,-7.196210e-03,0.468265,0.234132,Fail to reject H0
5,"H1: shelling, drone, and battle effects differ",shelling_count = drone_count = battle_count,NaN,0.000178,NaN,Reject H0
